# import library

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn #pip install scikit-learn
import scipy.stats as stats 
import csv
import requests
import statsmodels.api as sm
from statsmodels.formula.api import ols
import re

# option de visualisation output

In [ ]:
pd.set_option('display.max_columns', None) # affiche toutes les colonnes
pd.set_option('display.max_colwidth', None)  # Affiche les colonnes dans toute leur largeur
#pd.set_option('display.max_rows', None)# Affiche les lignes dans toute leur largeu

# Create df

In [ ]:
data=pd.read_csv("2016_Building_Energy_Benchmarking.csv")

# Explore

In [ ]:
data.head()

In [ ]:
data.shape

In [ ]:
data.info()

In [ ]:
data["DataYear"].unique()

In [ ]:
data["Outlier"].unique()

In [ ]:
data.columns

In [ ]:
data["Neighborhood"].unique()

In [ ]:
data["ZipCode"].unique()

In [ ]:
data.head()

## Create liste

In [ ]:
# keep only structural col
list_col_struct=["OSEBuildingID","PrimaryPropertyType","ZipCode","CouncilDistrictCode","YearBuilt","NumberofBuildings","NumberofFloors","PropertyGFATotal","PropertyGFAParking","PropertyGFABuilding(s)"
,"ListOfAllPropertyUseTypes",'LargestPropertyUseType', 'LargestPropertyUseTypeGFA',
       'SecondLargestPropertyUseType', 'SecondLargestPropertyUseTypeGFA',
       'ThirdLargestPropertyUseType', 'ThirdLargestPropertyUseTypeGFA']
len(list_col_struct)#17

In [ ]:
#check if "PrimaryPropertyType" and 'LargestPropertyUseType' are the same
set1=set(data["LargestPropertyUseType"]) # set guive only unique
set2=set(data["PrimaryPropertyType"])
difference1=set1.difference(set2)#Getting element present in list 1 that are not in 2
difference1# those col are not the same!

In [ ]:
# Create a short list
list_col_struct9 = ["OSEBuildingID","PrimaryPropertyType","CouncilDistrictCode","YearBuilt","NumberofBuildings",
"NumberofFloors","PropertyGFABuilding(s)",'LargestPropertyUseType', 'LargestPropertyUseTypeGFA']

In [ ]:
list_col_struct7 = ["OSEBuildingID",'LargestPropertyUseType',"CouncilDistrictCode","YearBuilt","NumberofBuildings",
"NumberofFloors",'LargestPropertyUseTypeGFA']

In [ ]:
data[list_col_struct7]

In [ ]:
# Create a df with those col
df_col_struct7=data.loc[:,data.columns.isin(list_col_struct7)]
df_col_struct7.head()

In [ ]:
# Create a df with those col
df_col_struct9=data.loc[:,data.columns.isin(list_col_struct9)]
df_col_struct9.head()

In [ ]:
data["BuildingType"].unique()

In [ ]:
motif = re.compile(r"(\w[Multifamily]{11}\d\w\w)")

In [ ]:

#re.search(motif, data["BuildingType"].unique() )
print(data[data.BuildingType.str.contains(r'[^Multifamily]')]["BuildingType"])
# MARCHE PAS, CHANGE DE STRAT

In [ ]:
print(data[data.BuildingType.str.contains(r'Multifamily\d\w\w')])

In [ ]:
# Drop Multifamily
#data.drop(data[data["BuildingType"] == 'Multifamily MR (5-9)'], inplace = True)
print(data[data["BuildingType"].str.contains("Multifamily")==True]["BuildingType"])# ok
data = data[data["BuildingType"].str.contains("Multifamily")==False]

In [ ]:
data["BuildingType"].unique() #ok

In [ ]:
data.shape

# Analyse Univariée on selected col (7)
## Description by col
### Description variable qualitative
#### index
##### Check doublons

In [ ]:
# Pas de doublons
df_col_struct7["OSEBuildingID"].duplicated().unique()

##### Nan values

In [ ]:
data[list_col_struct7].info()

#### Col Number of buildings

In [ ]:
# Num of building should be integer : problemes
data["NumberofBuildings"].unique()
# need to remove nan before changing type    

In [ ]:
# Distribution du nombre de building
plt.show()
fig=plt.figure(figsize=(20,10))
np.log(data["NumberofBuildings"].value_counts()).plot(kind='bar')
plt.title("Distribution du nombre de building", fontsize=15, fontweight="bold")
plt.xlabel('Nombre de building',fontsize=12, fontweight="bold")
plt.ylabel('Frequence absolue',fontsize=12, fontweight="bold")

# la plupart a un building
# pourquoi 0? a virer/ checker
# plot pas adapter seaborn plus lisible
# combien de 111? trop outlier, va compliquer les choses : remove si peu

In [ ]:
# Check what are the building with 0 building
data.loc[data["NumberofBuildings"] ==0, ["LargestPropertyUseType"]]
# Look like Nan...need to transform then in Nan

In [ ]:
# Replace 0 By Nan  (pas possible building a 0)
#data["NumberofBuildings"] = data[data["NumberofBuildings"] ==0].fillna()
data.replace(0, np.nan, inplace=True)
data[list_col_struct7] .info()#output:  3276 should be 3268
len(data[data["NumberofBuildings"].isna()]) #output should be 108 (100 zero + previous Nan (8))
#why 100 only?

In [ ]:
data = data.dropna(subset=["NumberofBuildings"])# remove row if nan in building col 

In [ ]:
data["NumberofBuildings"].unique()

In [ ]:
# change type
data["NumberofBuildings"] = data["NumberofBuildings"].astype(int)      
data["NumberofBuildings"].dtype

In [ ]:
data["NumberofBuildings"].unique()

In [ ]:
#df_col_struct7.loc[df_col_struct7["NumberofFloors"] > 100,:]
data.loc[data["NumberofBuildings"] == 111, ["LargestPropertyUseType"]]
# output : only one and non aberent
# Remove anyway because unique and will complicated the model !

In [ ]:
#remove outlier 111
data2=data[data["NumberofBuildings"] != 111]
data2["NumberofBuildings"].unique() 

In [ ]:
# Better representation: Distribution du nombre de building
plt.show()
fig=plt.figure(figsize=(20,10))
ybuilding = data2["NumberofBuildings"].value_counts()
ybuilding.plot(kind='bar')
plt.title("Distribution des complexes par nombre de batiments", fontsize=15, fontweight="bold")
plt.xlabel('Nombre de building',fontsize=12, fontweight="bold")
plt.ylabel('Frequence absolue',fontsize=12, fontweight="bold")
plt.ylim(0,60)
for index,val in enumerate(ybuilding[1:]):
    plt.text(x=index+1 , y = val+1 , s=f"{val}")

In [ ]:
ybuilding

In [ ]:
sns.barplot(x = "NumberofBuildings",
            y = np.log(data2["NumberofBuildings"]).value_counts(),
            data = data2)

# Show the plot
plt.show()


#### CouncilDistrictCode 

In [ ]:
data2[list_col_struct7].info()

In [ ]:
list_district = data2["CouncilDistrictCode"].unique()
list_district
#output:only 7

In [ ]:
#Distribution : pie plot
# % property
n_row_district = []
for el in list_district:
    n_row_district.append(len(data2[data2["CouncilDistrictCode"]== el ]))
n_row_district_tot = data2.shape[0]
list_perc_district = np.array(n_row_district)/n_row_district_tot

# create a pie chart
fig = plt.figure(figsize =(10, 7))
plt.pie(list_perc_district, labels = list_district, autopct='%1.0f%%')
plt.title("Distribution des complexes par quartier", fontsize=15, fontweight="bold")
# output too much modalite

In [ ]:
sum(list_perc_district)# verifie que sum=1

In [ ]:
list_district

In [ ]:
data2["CouncilDistrictCode"] = data2["CouncilDistrictCode"].astype(str)
# ou data2["CouncilDistrictCode"] = str(data2["CouncilDistrictCode"])
data2["CouncilDistrictCode"].dtypes 
# dtype('O') are string OK

#### NumberofFloors  

In [ ]:
data2["NumberofFloors"].unique()

In [ ]:
data2[list_col_struct7].info()

In [ ]:
#remove Nan because not a lot (14 and we can t replace them)
data2 = data2.dropna(subset=["NumberofFloors"])# remove row if nan in floor col 

In [ ]:
data2["NumberofFloors"].unique()

In [ ]:
# Change type
data2["NumberofFloors"] = data2["NumberofFloors"].astype(int)
# ou data2["CouncilDistrictCode"] = str(data2["CouncilDistrictCode"])
data2["NumberofFloors"].dtypes 
# output OK

In [ ]:
#Distribution
fig=plt.figure(figsize=(20,10))
yfloor = data2["NumberofFloors"].value_counts()
yfloor.plot(kind='bar')
plt.title("Distribution des batiments par nombre d etage", fontsize=15, fontweight="bold")
plt.xlabel('Nombre d etage',fontsize=12, fontweight="bold")
plt.ylabel('Frequence absolue',fontsize=12, fontweight="bold")
plt.ylim(0,60)


# 4 niveau car 0 RCH/groundfloor??? info, regarder def et tout collect de la meme facon? ou bien 2 facon de compter
# erreur dans la collect ou bien un RCH?? go info : PAS de 0 etage dans les 3275 lignes
# de grand batiments ou erreur? NON PAS ERREURS OK

# solrt

##### Recherche
le batiment le plus haut de Seatle possede 76 etages
19 des 20 plus haut building des US se trouvent à Seatle

#### YearBuilt 
LargestPropertyUseTypeGFA

In [ ]:
data2["YearBuilt"].unique()

In [ ]:
data2["Age"] = 2016 - data2["YearBuilt"]
data2["Age"] 

In [ ]:
fig=plt.figure(figsize=(20,10))
data2["Age"].hist(bins = 10)
plt.show()
# need categorisation 
# ne4ed plays with bins

In [ ]:
# Creation classe d'age (x6, tous les)20 ans...
# data2["Classe_Age"] = data2["Age"]//20
data2["Classe_Age"] = data2["Age"]//20 +1
data2["Classe_Age"].unique()
# output : plus classe eleve, plus ancien

In [ ]:
# Change type as object:
# Change type
data2["Classe_Age"] = data2["Classe_Age"].astype(str)
# ou data2["CouncilDistrictCode"] = str(data2["CouncilDistrictCode"])
data2["Classe_Age"].dtypes 
# output OK

In [ ]:
#Distribution : pie plot
list_classe_age = data2["Classe_Age"].unique()
# % property
n_row_ancien = []
for el in list_classe_age :
    n_row_ancien.append(len(data2[data2["Classe_Age"]== el ]))
list_perc_ancien = np.array(n_row_ancien)/data2.shape[0]

# create a pie chart
fig = plt.figure(figsize =(10, 7))
plt.pie(list_perc_ancien, labels = list_classe_age , autopct='%1.0f%%')
plt.title("Distribution des bâtiments par leur catégorie d'ancienneté", fontsize=15, fontweight="bold")
# 1 pour 20 ans

#### LargestPropertyUseType

In [ ]:
# create liste
list_property_type = data2["LargestPropertyUseType"].unique()
list_property_type

In [ ]:
# Look at Nan : if nothing in other col, remove row
data2[data2["LargestPropertyUseType"].isna()]
data2["LargestPropertyUseType"].fillna(data2["ListOfAllPropertyUseTypes"], inplace=True)

In [ ]:
data2[data2["LargestPropertyUseType"].isna()] #ok

In [ ]:
data2.shape #ok

In [ ]:
list_property_type

In [ ]:
data2.shape # I remove multifamily housing from BuildingType because they are residential...

In [ ]:
data2[data2["LargestPropertyUseType"] =='Multifamily Housing'] # check what are multi housing

In [ ]:
# Create Classes :

#listdelist = [Residence, hotel, parking, public, Commerce, Loisir, Office, Usine, Autres, Stockage, Data_Center, Sante, Education]
#dico
dico_bat = {"Residence" : ['Multifamily Housing','Other - Lodging/Residential'],
"hotel" : ['Hotel','Residence Hall/Dormitory'],
"parking" : ['Parking'],
"public" : ['Police Station','Other - Entertainment/Public Assembly','Social/Meeting Hall', 'Courthouse', 'Fire Station', 'Other - Services', 'Other - Public Services', 'Prison/Incarceration'],
"Commerce" : ['Restaurant', 'Other - Restaurant/Bar','Personal Services (Health/Beauty, Dry Cleaning, etc)', 'Lifestyle Center','Strip Mall','Automobile Dealership','Other - Mall','Retail Store','Repair Services (Vehicle, Shoe, Locksmith, etc)','Distribution Center','Supermarket/Grocery Store'],
"Loisir" : ['Library','Museum','Performing Arts', 'Movie Theater','Fitness Center/Health Club/Gym','Other - Recreation','Worship Facility', 'Convention Center'],
"Office" : ['Office','Financial Office','Bank Branch'],
"Usine" : ['Manufacturing/Industrial Plant'],
"Autres" : ['Other',  'Other - Utility'],
"Stockage" : ['Self-Storage Facility', 'Non-Refrigerated Warehouse', 'Refrigerated Warehouse'],
"Data_Center" : ['Data Center'],
"Sante": ['Residential Care Facility','Senior Care Community', 'Medical Office','Hospital (General Medical & Surgical)','Other/Specialty Hospital','Laboratory','Urgent Care/Clinic/Other Outpatient' ],
"Education" : ['K-12 School', 'College/University','Other - Education','Pre-school/Daycare','Adult Education']}

In [ ]:
data2['Classe_bat'] = 0
for k,v in dico_bat.items() :
    for el in v :
        #if data2['LargestPropertyUseType']== el:
        data2.loc[data2['LargestPropertyUseType']== el, ['Classe_bat']] = k
data2['Classe_bat'].unique()      

In [ ]:
data2[data2["Classe_bat"]== 0]

In [ ]:
data2["Classe_bat"].replace(to_replace = 0, value = "Various", inplace=True)

In [ ]:
data2["Classe_bat"].unique()

Visualisation graphique

In [ ]:
# % property
nbline=[]
for k in dico_bat.keys():
    nbline.append(len(data2[data2['Classe_bat']==k]))
nlinetot = data2.shape[0]
list_perc_type=np.array(nbline)/nlinetot
sum(list_perc_type)# verifie que sum=1

In [ ]:
# create a pie chart
fig = plt.figure(figsize =(10, 7))
plt.pie(list_perc_type, labels =  dico_bat.keys(), autopct='%1.0f%%')
plt.title("Distribution des types de propriétés non résidentielles", fontsize=15, fontweight="bold")
# output too much modalite
#pas le plus adapte

### Variables quantitatives

In [ ]:
#liste de variables quantitatives
list_col_struct9_quant = ["PropertyGFABuilding(s)",'LargestPropertyUseTypeGFA']

In [ ]:
# Create a df with those col
df_col_struct9_quant = data2.loc[:,data2.columns.isin(list_col_struct9_quant)]
df_col_struct9_quant.head()

In [ ]:
df_col_struct9_quant.describe().round(2)

In [ ]:
data2["LogPropertyGFABuilding"] = np.log(data2["PropertyGFABuilding(s)"]) 
data2["LogLargestPropertyUseTypeGFA"] = np.log(data2["LargestPropertyUseTypeGFA"])

In [ ]:
data2["LogLargestPropertyUseTypeGFA"].describe().round(2)

In [ ]:
data2["LogPropertyGFABuilding"].describe().round(2)

In [ ]:
# histogramme pour une col
fig=plt.figure(figsize=(20,10))
plt.title("PropertyGFABuilding(s)")
ax=plt.subplot(211)
sns.kdeplot(data2["PropertyGFABuilding(s)"])
#plt.yscale('log')
plt.subplot(212, sharex=ax)
#plt.hist(data2[data2["PropertyGFABuilding(s)"]])
plt.boxplot(data2["PropertyGFABuilding(s)"],vert=False)
plt.show()

In [ ]:
# histogramme pour une col
fig=plt.figure(figsize=(20,10))
plt.title("LargestPropertyUseTypeGFA")
ax=plt.subplot(211)
sns.kdeplot(data2["LargestPropertyUseTypeGFA"])
#plt.yscale('log')
plt.subplot(212, sharex=ax)
plt.boxplot(data2["LargestPropertyUseTypeGFA"], vert = False)
plt.show()

In [ ]:
#bar plot in the same grah for all float data:
sns.boxplot( data2[list_col_struct9_quant],orient="h",showmeans=True)
plt.ylabel('Category')
plt.xlabel('surface en pied carre')
plt.xlim(0,250_000)
plt.xticks(rotation=45)
plt.show()

In [ ]:
# New col : mean surface per etage (pondére par le nombre d'etage) de la totalite des proprietes
# Surface total / n bat * n etage moyen
data2["Surface_par_etage"] = (data2["PropertyGFABuilding(s)"] / (data2['NumberofBuildings']*data2['NumberofFloors'])).round(2)
data2["Surface_par_etage"]

#### Pour model 2 : secondary and terniary use type

### Variables quantitatives

In [ ]:
#liste de variables quantitatives
list_new_feature2 = ["LargestPropertyUseTypeGFA","SecondLargestPropertyUseTypeGFA","ThirdLargestPropertyUseTypeGFA"]

In [ ]:
# Create a df with those col
df_new_feature2_quant = data2.loc[:,data2.columns.isin(list_new_feature2)]
df_new_feature2_quant.head()

In [ ]:
df_new_feature2_quant.describe().round(2)

In [ ]:
# histogramme pour une col
fig=plt.figure(figsize=(20,10))
plt.title("SecondLargestPropertyUseTypeGFA")
ax=plt.subplot(211)
sns.kdeplot(data2["SecondLargestPropertyUseTypeGFA"])
plt.subplot(212, sharex=ax)
plt.boxplot(data2["SecondLargestPropertyUseTypeGFA"],vert=False)
plt.show()

In [ ]:
# histogramme pour une col
fig=plt.figure(figsize=(20,10))
plt.title("ThirdLargestPropertyUseTypeGFA")
ax=plt.subplot(211)
sns.kdeplot(data2["ThirdLargestPropertyUseTypeGFA"])
#plt.yscale('log')
plt.subplot(212, sharex=ax)
plt.boxplot(data2["ThirdLargestPropertyUseTypeGFA"], vert = False)
plt.show()

In [ ]:
#bar plot in the same grah for all float data:
sns.boxplot( df_new_feature2_quant,orient="h",showmeans=True)
plt.ylabel('Category')
plt.xlabel('surface en pied carre')
plt.xlim(0,250_000)
plt.xticks(rotation=45)
plt.show()

In [ ]:
data2["LogSecondLargestPropertyUseTypeGFA"] = np.log(data2["SecondLargestPropertyUseTypeGFA"])
data2["LogThirdLargestPropertyUseTypeGFA"] = np.log(data2["ThirdLargestPropertyUseTypeGFA"])

# analyse bivarie

In [ ]:
data2['NumberofBuildings'].unique()

In [ ]:
# On affiche le nuage de points nombre de building vs surface
plt.plot(data2['NumberofBuildings'],data2['LargestPropertyUseTypeGFA'], 'ro', markersize=4)
plt.show()
#virer le 110 mettre de cote pour ce graph
# tellement loin du reste /virer exeption donc va pourrir mon model
#but etre plus proche de TOUTES les DATAS
#penaliser model pour UNe seule valeur
# pas vraiment pertinent comme comparaison. il faudrait faire par nombre de buildiong

In [ ]:
# On affiche le nuage de points nombre d etage vs surface
plt.plot(data2['NumberofFloors'],data2['LargestPropertyUseTypeGFA'], 'ro', markersize=4)
plt.show()
# nombre d etage plus de 100??? NON
# Remove outlier

In [ ]:
# On affiche le nuage de points age vs surface
plt.plot(data2['Age'],data2['LargestPropertyUseTypeGFA'], 'ro', markersize=4)
plt.show()

In [ ]:
# On affiche le nuage de points age vs surface
plt.plot(data2['Age'],data2["NumberofFloors"], 'ro', markersize=4)
plt.show()
# pas de tendance entre l age du batiment et le nombre d'etage

In [ ]:
#data2.columns
list_col_struct_model1 = [
 'LargestPropertyUseType',
 'CouncilDistrictCode',
 'Classe_Age',
 'NumberofBuildings',
 'NumberofFloors',
 'LargestPropertyUseTypeGFA','PropertyGFABuilding(s)']

In [ ]:
corrMatrix = data2[list_col_struct_model1].corr()
corrMatrix

In [ ]:
sns.heatmap(corrMatrix, annot=True)
plt.show()

In [ ]:
sns.pairplot(data2,vars= ['NumberofBuildings','NumberofFloors','LargestPropertyUseTypeGFA','PropertyGFABuilding(s)'], hue="Classe_Age")
plt.show()

In [ ]:
 # Per disctrict
sns.pairplot(data2,vars= ['NumberofBuildings','NumberofFloors','LargestPropertyUseTypeGFA','PropertyGFABuilding(s)'], hue='CouncilDistrictCode')
plt.show()

In [ ]:
# Per larsgest used type
sns.pairplot(data2,vars= ['NumberofBuildings','NumberofFloors','LargestPropertyUseTypeGFA','PropertyGFABuilding(s)'], hue="Classe_bat")
plt.show()

In [ ]:
# make sure that the featureas are not correlated (CHECK MORE)
list_tempo =  ['CouncilDistrictCode',
 'Classe_Age',
 'NumberofBuildings',
 'NumberofFloors',
 'PropertyGFABuilding(s)',"GHGEmissionsIntensity" ]

In [ ]:
corrMatrix=data2[list_tempo].corr()
corrMatrix
sns.heatmap(corrMatrix, annot=True)
plt.show()

In [ ]:
 # Per larsgest used type
sns.pairplot(data2,vars= [
 'YearBuilt',
 'NumberofBuildings',
 'NumberofFloors',
 'PropertyGFABuilding(s)'], hue="GHGEmissionsIntensity" )
plt.show()

In [ ]:
list_col_struct2 = [
 'PrimaryPropertyType',
 'Latitude', 'Longitude',
 'CouncilDistrictCode',
 'YearBuilt',
 'NumberofBuildings',
 'NumberofFloors',
 'PropertyGFATotal',
 'PropertyGFAParking',
 'PropertyGFABuilding(s)',
 'ListOfAllPropertyUseTypes',
 'LargestPropertyUseType',
 'LargestPropertyUseTypeGFA',
 'SecondLargestPropertyUseType',
 'SecondLargestPropertyUseTypeGFA',
 'ThirdLargestPropertyUseType',
 'ThirdLargestPropertyUseTypeGFA',
       'Classe_Age', 'Classe_bat', 'Surface_par_etage', 'GHGEmissionsIntensity','Age']

In [ ]:
list_col_struct

In [ ]:
# Create a liste with y (variable to test)
list_corr_struct_GHE1 = [
 'PrimaryPropertyType',
 'Age',
 'Surface_par_etage',
 'NumberofBuildings',
 'NumberofFloors',
 'PropertyGFATotal',
 'PropertyGFAParking',
 'PropertyGFABuilding(s)',
 'ListOfAllPropertyUseTypes',
 'LargestPropertyUseType',
 'LargestPropertyUseTypeGFA',
 'SecondLargestPropertyUseType',
 'SecondLargestPropertyUseTypeGFA',
 'ThirdLargestPropertyUseType',
 'ThirdLargestPropertyUseTypeGFA',
 'GHGEmissionsIntensity']

In [ ]:
corrMatrix22=data2[list_corr_struct_GHE1].corr()# car considere zipcode comme variable quantitative alors que qualitative
corrMatrix22
sns.heatmap(corrMatrix22, annot=True)
plt.show()

In [ ]:
corrMatrix23=data2[list_col_struct2[5:-1]].corr()
corrMatrix23
sns.heatmap(corrMatrix23, annot = True)

plt.show()

In [ ]:
list_struct_Star_CO2 = [
 'NumberofBuildings',
 'NumberofFloors',
 'PropertyGFABuilding(s)',
 'LargestPropertyUseTypeGFA',
 'SecondLargestPropertyUseTypeGFA',
 'ThirdLargestPropertyUseTypeGFA', 
 'ENERGYSTARScore', 'GHGEmissionsIntensity']

In [ ]:
corrMatrix24=data2[list_struct_Star_CO2].corr()
corrMatrix24
sns.heatmap(corrMatrix24, annot = True)

plt.show()

### Analysis vs Electricity

In [ ]:
list_struct_NRJ = [
 'NumberofBuildings',
 'NumberofFloors',
 'PropertyGFATotal',
 'PropertyGFAParking',
 'PropertyGFABuilding(s)',
 'LargestPropertyUseTypeGFA',
 'SecondLargestPropertyUseTypeGFA',
 'ThirdLargestPropertyUseTypeGFA', 
 'ENERGYSTARScore',
'SiteEUIWN(kBtu/sf)', 'SiteEnergyUseWN(kBtu)', 'Electricity(kBtu)']

In [ ]:
corrMatrix22=data2[list_struct_NRJ].corr()# car considere zipcode comme variable quantitative alors que qualitative
corrMatrix22
sns.heatmap(corrMatrix22, annot=True)
plt.show()

In [ ]:
#Export DF
data2.to_csv("data2.csv",index =False)